# 🎯 Lesson 13: Interview Copilot — Foundation

## Building Your Personal DSA Interview Coach

This is where everything comes together. Over lessons 13-15, we build a production-ready interview preparation agent.

**Architecture:** Multi-agent | Voice+Text | Code Sandbox | Memory | Dashboard

In [ ]:
# !pip install pyautogen openai chromadb streamlit python-dotenv rich pydantic
import os, json
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
load_dotenv()
console = Console()
Path('./copilot_data').mkdir(exist_ok=True)
console.print('[bold green]🚀 Interview Copilot — Initializing...[/bold green]')

## 📐 Data Models

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List
from datetime import datetime

@dataclass
class DSAProblem:
    id: str
    title: str
    topic: str
    difficulty: str  # easy / medium / hard
    description: str
    examples: List[dict]
    constraints: List[str]
    hints: List[str]  # 3 levels: nudge, guide, explain
    solution_python: str
    time_complexity: str
    space_complexity: str
    leetcode_url: str

@dataclass
class Attempt:
    problem_id: str
    user_code: str
    hints_used: int
    time_taken_seconds: int
    passed: bool
    score: float  # 0.0 - 1.0
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

@dataclass
class Session:
    session_id: str
    user_name: str
    start_time: str = field(default_factory=lambda: datetime.now().isoformat())
    attempts: List[Attempt] = field(default_factory=list)
    weak_topics: List[str] = field(default_factory=list)

    def success_rate(self) -> float:
        if not self.attempts: return 0.0
        return sum(1 for a in self.attempts if a.passed) / len(self.attempts)

console.print('[green]✅ Data models defined[/green]')

## 📚 DSA Problem Bank

In [ ]:
PROBLEM_BANK = [
    DSAProblem(
        id='001',
        title='Two Sum',
        topic='Arrays',
        difficulty='easy',
        description='Given array nums and target, return indices of two numbers that add up to target.',
        examples=[{'input': 'nums=[2,7,11,15], target=9', 'output': '[0,1]'}],
        constraints=['2 <= nums.length <= 10^4', 'Exactly one solution exists'],
        hints=[
            'Think about what complement you need for each number.',
            'Can you store previously seen numbers somewhere for O(1) lookup?',
            'Use a hashmap: for each num, check if target-num exists in the map.'
        ],
        solution_python='def twoSum(nums, target):\n    seen = {}\n    for i, n in enumerate(nums):\n        if target-n in seen:\n            return [seen[target-n], i]\n        seen[n] = i',
        time_complexity='O(n)',
        space_complexity='O(n)',
        leetcode_url='https://leetcode.com/problems/two-sum/'
    ),
    DSAProblem(
        id='002',
        title='Valid Parentheses',
        topic='Stack',
        difficulty='easy',
        description='Given string s with brackets, determine if it is valid (properly closed in order).',
        examples=[{'input': 's="()[]{}"', 'output': 'True'}],
        constraints=['1 <= s.length <= 10^4', 's consists of brackets only'],
        hints=[
            'What data structure is great for tracking nested structures?',
            'When you see a closing bracket, what should the last opened bracket be?',
            'Use a stack: push open brackets, pop and check when you see a closing bracket.'
        ],
        solution_python='def isValid(s):\n    stack = []\n    pairs = {")": "(", "]": "[", "}": "{"}\n    for c in s:\n        if c in "([{":\n            stack.append(c)\n        elif not stack or stack[-1] != pairs[c]:\n            return False\n        else:\n            stack.pop()\n    return len(stack) == 0',
        time_complexity='O(n)',
        space_complexity='O(n)',
        leetcode_url='https://leetcode.com/problems/valid-parentheses/'
    ),
    DSAProblem(
        id='003',
        title='Binary Search',
        topic='Binary Search',
        difficulty='easy',
        description='Given sorted array nums and target, return index of target or -1 if not found.',
        examples=[{'input': 'nums=[-1,0,3,5,9,12], target=9', 'output': '4'}],
        constraints=['1 <= nums.length <= 10^4', 'nums is sorted in ascending order'],
        hints=[
            'The array is sorted — can you eliminate half the possibilities each step?',
            'Use two pointers: left and right. What do you check at the middle?',
            'If mid == target return. If mid < target, search right half. Else search left half.'
        ],
        solution_python='def search(nums, target):\n    l, r = 0, len(nums)-1\n    while l <= r:\n        mid = (l+r)//2\n        if nums[mid] == target: return mid\n        elif nums[mid] < target: l = mid+1\n        else: r = mid-1\n    return -1',
        time_complexity='O(log n)',
        space_complexity='O(1)',
        leetcode_url='https://leetcode.com/problems/binary-search/'
    ),
]

console.print(f'[green]✅ Problem bank loaded: {len(PROBLEM_BANK)} problems[/green]')

# Display problem bank
table = Table(title='DSA Problem Bank')
table.add_column('ID', style='cyan')
table.add_column('Title', style='white')
table.add_column('Topic', style='yellow')
table.add_column('Difficulty', style='green')
table.add_column('LeetCode', style='blue')
for p in PROBLEM_BANK:
    table.add_row(p.id, p.title, p.topic, p.difficulty, p.leetcode_url)
console.print(table)

## 🔧 AutoGen Configuration for Copilot

In [ ]:
from autogen import AssistantAgent, UserProxyAgent

llm_config = {
    'config_list': [{
        'model': os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o'),
        'api_type': 'azure',
        'api_key': os.getenv('AZURE_OPENAI_API_KEY'),
        'base_url': os.getenv('AZURE_OPENAI_ENDPOINT'),
        'api_version': os.getenv('AZURE_OPENAI_API_VERSION'),
    }],
    'temperature': 0
}

def create_interviewer_agent(problem: DSAProblem) -> AssistantAgent:
    return AssistantAgent(
        name='Interviewer',
        llm_config=llm_config,
        system_message=f"""You are a friendly but rigorous technical interviewer at a top tech company.
Current Problem: {problem.title}
Topic: {problem.topic} | Difficulty: {problem.difficulty}

Description: {problem.description}
Examples: {json.dumps(problem.examples)}
Constraints: {', '.join(problem.constraints)}

Your job:
1. Present the problem clearly
2. Ask clarifying questions if the user seems stuck
3. Evaluate their approach and code
4. Give a score 1-10 at the end
5. Say TERMINATE when the session should end

DO NOT give the solution directly. Guide with questions."""
    )

def create_hint_agent(problem: DSAProblem) -> AssistantAgent:
    return AssistantAgent(
        name='HintGiver',
        llm_config=llm_config,
        system_message=f"""You are a supportive coding mentor who gives hints without spoiling.
Problem: {problem.title}
Hints available (give in order, only when asked):
- Level 1 (nudge): {problem.hints[0]}
- Level 2 (guide): {problem.hints[1]}
- Level 3 (explain): {problem.hints[2]}

Track how many hints have been given. Never give Level 3 before Level 1 and 2.
Be encouraging and never make the user feel bad for needing hints."""
    )

console.print('[green]✅ Agent factory functions ready![/green]')

## 🧪 Test the Foundation

In [ ]:
# Pick a problem and test the interviewer
selected_problem = PROBLEM_BANK[0]  # Two Sum
console.print(Panel(
    f'[bold]Problem: {selected_problem.title}[/bold]\n'
    f'Topic: {selected_problem.topic} | Difficulty: {selected_problem.difficulty}\n'
    f'URL: {selected_problem.leetcode_url}',
    title='🎯 Selected Problem'
))

interviewer = create_interviewer_agent(selected_problem)
hint_agent = create_hint_agent(selected_problem)

# Quick test — user sends their approach
student = UserProxyAgent(
    name='Avnish',
    human_input_mode='NEVER',
    max_consecutive_auto_reply=2,
    is_termination_msg=lambda x: 'TERMINATE' in x.get('content', '')
)

student.initiate_chat(
    interviewer,
    message="I'll use a hashmap to solve Two Sum. For each number, I'll store it and its index, then check if target-num exists. Let me code it."
)

## ✅ Foundation Built!
Next: Lesson 14 adds the full code sandbox, voice input, and complete session flow.